> **Public-repository preparation copy.** This notebook is based on the corresponding final-run notebook used for the study. 
Execution outputs were cleared for publication. Set the `QA_DATASET_CSV` environment variable (or edit the default path below) 
to the location of `finetune_dataset_fixed_3000.csv` before running. Do not place access tokens in notebook source.


# Prepration

Memory and Dtype Issues: Early reports indicated memory and dtype mismatch errors with DoRA in Unsloth, often linked to the flash_attn package. Ensure flash_attn is uninstalled (pip uninstall flash-attn) to avoid issues.

In [ ]:
!pip install pip3-autoremove
!pip install torch==2.10.0+cu128 xformers==0.0.35
!pip install unsloth==2026.5.4
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install Triton==3.6.0

In [ ]:
import unsloth
from unsloth import FastLanguageModel, unsloth_train, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
import torch
import gc
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

# Model loading

In [ ]:
# Load 4-bit quantized pretrained LLaMA 3.1 model from Unsloth
model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    load_in_4bit=True,
    device_map={'': torch.cuda.current_device()},
# Hugging Face access tokens must be supplied securely; never hard-code them here.
)

In [ ]:
# Apply LoRA + DoRA adaptation to selected modules for efficient fine-tuning

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
    use_dora=True,
)

# Dataset loading and fixing format

In [ ]:
# Define a consistent system prompt to guide assistant behavior
#SYSTEM_PROMPT = """
#You are a helpful AI assistant.
#Use the provided context to answer accurately and clearly.
#If multiple questions are asked, answer each separately.
#"""

SYSTEM_PROMPT = """You are a helpful AI assistant.

When a user provides context, use only that context to answer their question.
If the question includes multiple sub-questions, answer each one concisely on a new line, in the same order.

Always strive to provide clear, structured, and accurate answers."""

In [ ]:
# Use proper chat template for LLaMA 3.1 formatting
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

In [ ]:
# Load dataset containing input/output pairs and split indicators
import pandas as pd
import os
DATASET_CSV_PATH = os.environ.get('QA_DATASET_CSV', '/kaggle/input/datasets/lemika/finetune-dataset-fixed-3000-final-form/finetune_dataset_fixed_3000.csv')
dataset = pd.read_csv(DATASET_CSV_PATH)

In [ ]:
train_texts, validation_texts = [], []

for _, row in dataset.iterrows():
    split = row.get("split", "").strip()
    context = str(row.get("context") or "")
    questions = str(row.get("question") or "")
    answers = str(row.get("answer") or "")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {context}\nQuestion(s):\n{questions}"},
        {"role": "assistant", "content": answers}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    if split == "train":
        train_texts.append(text)
    elif split == "validation":
        validation_texts.append(text)

train_dataset = Dataset.from_dict({"text": train_texts})
validation_dataset = Dataset.from_dict({"text": validation_texts})

In [ ]:
# Preview dataset samples
train_dataset['text'][0:3]

In [ ]:
# Preview dataset samples
validation_dataset['text'][0:3]

In [ ]:
# Preview dataset samples
print(train_dataset[:5])

In [ ]:
lengths = [
    len(tokenizer(x)["input_ids"])
    for x in train_texts
]

print(pd.Series(lengths).describe(percentiles=[0.5,0.9,0.95,0.99]))

# Training and preprations

In [ ]:
# Define training configuration parameters
training_args=TrainingArguments(
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    warmup_ratio = 0.05,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="./outputs",
    logging_dir="./logs",
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_total_limit = 2,
    report_to = "tensorboard",
)

In [ ]:
import torch
import time
import csv
import os

# ---------------------------------------------------------
# GPU memory tracking configuration
# ---------------------------------------------------------
assert torch.cuda.is_available(), "CUDA GPU is not available."

device = torch.cuda.current_device()
gpu_name = torch.cuda.get_device_name(device)

# Reset peak memory statistics before the measured run
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

start_time = time.time()

print(f"GPU: {gpu_name}")
print(f"GPU index: {device}")
print("Starting training...\n")

In [ ]:
# Initialize training logs
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    dataset_text_field="text", # Explicitly point to the formatted column
    max_seq_length=2048,
    args=training_args,
    callbacks=[EarlyStoppingCallback(1)],
    packing=False,
)

trainer_stats = unsloth_train(trainer)

# Collect logs
log_history = trainer.state.log_history

# Save model checkpoint and tokenizer to outputs
print("------------------------------------------")
print("saving model ...")
trainer.save_model("/kaggle/working/fine_tuned_model")
tokenizer.save_pretrained("/kaggle/working/fine_tuned_model")


# Function to clear memory between training rounds
def clean_memory():
  torch.cuda.synchronize()
  torch.cuda.empty_cache()
  gc.collect()

clean_memory()

In [ ]:
# Synchronize so all CUDA operations finish before measuring
torch.cuda.synchronize()

elapsed_time = time.time() - start_time

# Peak memory statistics
peak_allocated = torch.cuda.max_memory_allocated(device)
peak_reserved = torch.cuda.max_memory_reserved(device)

# Current memory statistics
current_allocated = torch.cuda.memory_allocated(device)
current_reserved = torch.cuda.memory_reserved(device)

def gb(x):
    return x / (1024 ** 3)

print("\n" + "=" * 60)
print("GPU MEMORY REPORT")
print("=" * 60)
print(f"GPU:                    {gpu_name}")
print(f"Training time:          {elapsed_time / 3600:.2f} h")
print(f"Peak allocated memory:  {gb(peak_allocated):.3f} GB")
print(f"Peak reserved memory:   {gb(peak_reserved):.3f} GB")
print(f"Final allocated memory: {gb(current_allocated):.3f} GB")
print(f"Final reserved memory:  {gb(current_reserved):.3f} GB")
print("=" * 60)

In [ ]:
# ---------------------------------------------------------
# Save result to CSV
# ---------------------------------------------------------
log_file = "gpu_memory_log.csv"

file_exists = os.path.exists(log_file)

with open(log_file, "a", newline="") as f:
    writer = csv.writer(f)

    if not file_exists:
        writer.writerow([
            "gpu",
            "training_time_sec",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "final_allocated_gb",
            "final_reserved_gb"
        ])

    writer.writerow([
        gpu_name,
        round(elapsed_time, 2),
        round(gb(peak_allocated), 3),
        round(gb(peak_reserved), 3),
        round(gb(current_allocated), 3),
        round(gb(current_reserved), 3)
    ])

print(f"\nSaved to: {log_file}")

In [ ]:
# Convert logs directly (single run)
training_logs = pd.DataFrame(log_history)

# Cleaned and split DataFrames
train_loss_logs = training_logs[training_logs['train_loss'].notna()].reset_index(drop=True)
eval_loss_logs = training_logs[training_logs['eval_loss'].notna()].reset_index(drop=True)
train_metrics_logs = training_logs[training_logs['loss'].notna()].reset_index(drop=True)

# Sanity check
print("Train loss logs:", train_loss_logs.shape)
print("Eval loss logs:", eval_loss_logs.shape)
print("Train step logs:", train_metrics_logs.shape)

# Save logs
import os
output_dir = "/kaggle/working/fine_tune_logs"
os.makedirs(output_dir, exist_ok=True)

#train_loss_logs.to_csv(f"{output_dir}/train_loss_logs.csv", index=False)
eval_loss_logs.to_csv(f"{output_dir}/eval_loss_logs.csv", index=False)
train_metrics_logs.to_csv(f"{output_dir}/train_metrics_logs.csv", index=False)

# Plot
import matplotlib.pyplot as plt

x_axis = 'epoch' if 'epoch' in training_logs.columns else 'step'

plt.figure(figsize=(10, 4))

# Use step-based training loss
plt.plot(train_metrics_logs[x_axis], train_metrics_logs['loss'], label="Train Loss")

# Eval stays the same
plt.plot(eval_loss_logs[x_axis], eval_loss_logs['eval_loss'], label="Eval Loss")

plt.xlabel(x_axis.capitalize())
plt.ylabel("Loss")
plt.title("Training and Evaluation Loss")
plt.legend()
plt.grid(True)
plt.show()